# Medical VLM cross-panel benchmark (Kaggle)

Runs MedGemma, Lingshu, and MedVLM-R1 on both held-out compositional-VQA tracks. Predictions checkpoint under `/kaggle/working/womens_health_medvlm_results` and the final cell creates a downloadable ZIP.

Before running: choose **Accelerator: GPU T4 x2**, enable **Internet**, and add a Kaggle secret named `HF_TOKEN`. MedGemma access must already be accepted for that Hugging Face account. This is a supplemental direct-VLM experiment; the original Stage 11 design's pre-existing-prediction baseline remains separate.

In [ ]:
import os, subprocess, sys
from pathlib import Path

WORK = Path('/kaggle/working')
REPO = WORK / 'womens-health-project'
ROOT = WORK / 'womens_health_medvlm'
PIPELINE = ROOT / 'drive_archive'
RESULTS = WORK / 'womens_health_medvlm_results'
RESULTS.mkdir(parents=True, exist_ok=True)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print(gpu.stdout or gpu.stderr)
if gpu.returncode != 0:
    raise RuntimeError('No GPU detected. Enable GPU T4 x2 in Kaggle notebook settings.')

In [ ]:
repo_url = 'https://github.com/Fasih20/Curated-Multimodal-Dataset-for-Women-Health-Imaging.git'
if (REPO / '.git').exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', repo_url, str(REPO)], check=True)
os.chdir(REPO)
print('Commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                'gdown', 'pyarrow', 'scipy', 'transformers>=4.53,<5',
                'accelerate', 'bitsandbytes', 'qwen-vl-utils'], check=True)
print('Dependencies installed.')

In [ ]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret('HF_TOKEN')
if not token:
    raise RuntimeError('Add HF_TOKEN under Add-ons > Secrets and enable it for this notebook.')
os.environ['HF_TOKEN'] = token
print('HF_TOKEN loaded (value hidden).')

In [ ]:
import zipfile
import gdown

FILE_ID = '1hKgZ3jNz6NRrlPEuNg8LBTXiX3kMtphX'
ARCHIVE = ROOT / 'pipeline_data.zip'
MARKER = PIPELINE / '.extraction_complete'
ROOT.mkdir(parents=True, exist_ok=True)

if not ARCHIVE.exists() or not zipfile.is_zipfile(ARCHIVE):
    gdown.download(id=FILE_ID, output=str(ARCHIVE), quiet=False)
if not ARCHIVE.exists() or not zipfile.is_zipfile(ARCHIVE):
    raise RuntimeError(f'Download is not a valid ZIP: {ARCHIVE}')

if not MARKER.exists():
    PIPELINE.mkdir(parents=True, exist_ok=True)
    base = PIPELINE.resolve()
    with zipfile.ZipFile(ARCHIVE) as zf:
        for member in zf.infolist():
            target = (PIPELINE / member.filename).resolve()
            if target != base and base not in target.parents:
                raise RuntimeError(f'Unsafe ZIP member: {member.filename!r}')
        print(f'Extracting {len(zf.infolist()):,} entries...')
        zf.extractall(PIPELINE)
    MARKER.write_text('ok\n')
print('Pipeline root:', PIPELINE)
print('Panel manifest present:', (PIPELINE / 'panels_v1/panel_manifest.csv').exists())

In [ ]:
env = os.environ.copy()
env['PIPELINE_ROOT'] = str(PIPELINE)
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
stage = REPO / '11_compositional_panel_vqa'
for script in ['01_fix_panel_caption_alignment.py', '02_generate_compositional_questions.py']:
    print('Running', script)
    subprocess.run([sys.executable, str(stage / script), '--track', 'both'], check=True, env=env)
print('Both evaluation datasets are ready.')

In [ ]:
import shutil

out = PIPELINE / 'compositional_vqa_v1'
drive_predictions = RESULTS / 'medical_vlm_predictions'
drive_predictions.mkdir(parents=True, exist_ok=True)
local_predictions = out / 'medical_vlm_predictions'
if local_predictions.exists() and not local_predictions.is_symlink():
    shutil.copytree(local_predictions, drive_predictions, dirs_exist_ok=True)
    shutil.rmtree(local_predictions)
if not local_predictions.exists() and not local_predictions.is_symlink():
    local_predictions.symlink_to(drive_predictions, target_is_directory=True)
metrics_link = out / 'medical_vlm_metrics.json'
results_metrics = RESULTS / 'medical_vlm_metrics.json'
if not metrics_link.exists() and not metrics_link.is_symlink():
    metrics_link.symlink_to(results_metrics)
print('Checkpoint directory:', drive_predictions)

## Run the six resumable evaluations

Each model/track runs in a separate process, releasing both GPUs before the next load. Prediction Parquets are checkpointed every 10 questions. If the session is interrupted, rerun the notebook cells; completed questions are skipped.

In [ ]:
models = ['medgemma-4b-it', 'lingshu-7b', 'medvlm-r1']
tracks = ['clip', 'biomedclip']
benchmark = stage / '07_benchmark_medical_vlms.py'
for model in models:
    for track in tracks:
        print(f'\n===== {model} / {track} =====', flush=True)
        subprocess.run([sys.executable, str(benchmark), '--model', model, '--track', track],
                       check=True, env=env)
print('All medical-VLM evaluations completed.')

In [ ]:
import json
import pandas as pd
from IPython.display import display

metrics = json.loads(results_metrics.read_text())
summary = pd.DataFrame([{
    'model': row['model'], 'track': row['track'], 'n': row['n'],
    'accuracy': row['accuracy']
} for row in metrics]).sort_values(['model', 'track'])
display(summary.style.format({'accuracy': '{:.3%}'}))
summary.to_csv(RESULTS / 'medical_vlm_summary.csv', index=False)
for artifact in out.glob('compositional_vqa_dataset_*.parquet'):
    shutil.copy2(artifact, RESULTS / artifact.name)
archive = shutil.make_archive(str(WORK / 'womens_health_medvlm_results'), 'zip', RESULTS)
print('Download from the Kaggle Files pane:', archive)